In [1]:
import xarray as xr
import numpy as np
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature

In [12]:
ds_m2 = xr.open_dataset("G:/TPXO10_atlas_v2_nc-selected/h_m2_tpxo10_atlas_30_v2.nc")
ds_s2 = xr.open_dataset("G:/TPXO10_atlas_v2_nc-selected/h_s2_tpxo10_atlas_30_v2.nc")

In [19]:
grid_ds.variables

{'lon_z': <class 'netCDF4.Variable'>
 float64 lon_z(nx)
     long_name: longitude of Z nodes
     units: degree_east
 unlimited dimensions: 
 current shape = (10800,)
 filling on, default _FillValue of 9.969209968386869e+36 used,
 'lat_z': <class 'netCDF4.Variable'>
 float64 lat_z(ny)
     long_name: latitude of Z nodes
     units: degree_north
 unlimited dimensions: 
 current shape = (5401,)
 filling on, default _FillValue of 9.969209968386869e+36 used,
 'lon_u': <class 'netCDF4.Variable'>
 float64 lon_u(nx)
     long_name: longitude of U nodes
     units: degree_east
 unlimited dimensions: 
 current shape = (10800,)
 filling on, default _FillValue of 9.969209968386869e+36 used,
 'lat_u': <class 'netCDF4.Variable'>
 float64 lat_u(ny)
     long_name: latitude of U nodes
     units: degree_north
 unlimited dimensions: 
 current shape = (5401,)
 filling on, default _FillValue of 9.969209968386869e+36 used,
 'lon_v': <class 'netCDF4.Variable'>
 float64 lon_v(nx)
     long_name: longitude 

In [30]:
with rasterio.open(
    "G:\Tidal_range\jiangsu_tidal_range_2021.tif",
    "w",
    driver="GTiff",
    height=tide_range.shape[0],
    width=tide_range.shape[1],
    count=1,
    dtype="float32",
    crs="EPSG:4326",
    transform=transform,
    nodata=np.nan,
) as dst:
    dst.write(tide_range, 1)

print("✅ 成功生成 GeoTIFF 文件：G:\Tidal_range\jiangsu_tidal_range_2021.tif")


✅ 成功生成 GeoTIFF 文件：G:\Tidal_range\jiangsu_tidal_range_2021.tif


In [1]:
import numpy as np
import netCDF4 as nc
import pandas as pd
import rasterio
from rasterio.transform import from_origin
from affine import Affine

# === 1. 读取网格 ===
grid_ds = nc.Dataset("G:/TPXO10_atlas_v2_nc-selected/grid_tpxo10atlas_v2.nc")
lon = grid_ds.variables["lon_z"][:]
lat = grid_ds.variables["lat_z"][:]

# 创建 meshgrid，注意 indexing='ij' 保证与数据对齐
lon2d, lat2d = np.meshgrid(lon, lat, indexing='ij')  # shape: (nx, ny)
res = lon[1] - lon[0]

# 使用 affine 明确向下递减（纬度从北向南）
transform = Affine.translation(lon.min(), lat.max()) * Affine.scale(res, -res)

# === 2. 读取 M2 和 S2 分潮的 hRe 和 hIm ===
def read_complex_component(filename):
    ds = nc.Dataset(filename)
    hRe = ds.variables['hRe'][:]
    hIm = ds.variables['hIm'][:]
    amp = np.sqrt(hRe**2 + hIm**2)
    phase = np.arctan2(-hIm, hRe) * 180 / np.pi
    return amp, phase

amp_m2, pha_m2 = read_complex_component("G:/TPXO10_atlas_v2_nc-selected/h_m2_tpxo10_atlas_30_v2.nc")
amp_s2, pha_s2 = read_complex_component("G:/TPXO10_atlas_v2_nc-selected/h_s2_tpxo10_atlas_30_v2.nc")

# === 3. 构建每日时间序列（减少运算量）===
start_time = pd.Timestamp("2021-01-01 00:00:00")
end_time = pd.Timestamp("2021-02-28 00:00:00")
time_series = pd.date_range(start=start_time, end=end_time, freq='D')  # 每天一次
time_hours = (time_series - start_time).total_seconds() / 3600.0  # 小时为单位

print('len(time_hours)', len(time_hours))

omega_m2 = 28.9841042  # M2 分潮频率（°/hr）
omega_s2 = 30.0        # S2 分潮频率（°/hr）

ny, nx = amp_m2.shape
tide_series = np.zeros((ny, nx, len(time_hours)), dtype=np.float32)

print("🔄 正在重建每日潮位时间序列...")
for i, th in enumerate(time_hours):
    tide_series[:, :, i] = (
        amp_m2 * np.cos(np.deg2rad(omega_m2 * th - pha_m2)) +
        amp_s2 * np.cos(np.deg2rad(omega_s2 * th - pha_s2))
    )

# === 4. 计算潮差（两个月内最大 - 最小） ===
tide_range = tide_series.max(axis=-1) - tide_series.min(axis=-1)  # shape: (ny, nx)





# # === 5. 输出为 GeoTIFF ===
# output_path = "G:/Tidal_range/jiangsu_tidal_range_2021A.tif"
# with rasterio.open(
#     output_path,
#     "w",
#     driver="GTiff",
#     height=tide_range.shape[0],
#     width=tide_range.shape[1],
#     count=1,
#     dtype="float32",
#     crs="EPSG:4326",
#     transform=transform,
#     nodata=np.nan,
# ) as dst:
#     dst.write(tide_range, 1)

# print(f"✅ 成功生成 GeoTIFF 文件：{output_path}")


len(time_hours) 59
🔄 正在重建每日潮位时间序列...


In [2]:
tide_range_mean = np.mean(tide_series, axis=-1)
tide_max =  tide_series.max(axis=-1) 
tide_min =  tide_series.min(axis=-1) 
tide_range_mean.shape

(10800, 5401)

In [5]:
from rasterio.transform import from_origin
import numpy as np

# 转置 + 上下翻转
tide_range_to_write = np.flipud(tide_range.T)

# 创建 transform（左上角为起点，纬度递减）
transform = from_origin(lon.min(), lat.max(), res, res)

with rasterio.open(
    "G:/Tidal_range/tide_range_2021.tif",
    "w",
    driver="GTiff",
    height=tide_range_to_write.shape[0],  # 5401
    width=tide_range_to_write.shape[1],   # 10800
    count=1,
    dtype="float32",
    crs="EPSG:4326",
    transform=transform,
    nodata=np.nan,
) as dst:
    dst.write(tide_range_to_write, 1)

print("✅ 输出 shape:", tide_range_to_write.shape)
print("✅ GeoTIFF 写入完成，方向正常，ArcGIS 应为横向展开")


✅ 输出 shape: (5401, 10800)
✅ GeoTIFF 写入完成，方向正常，ArcGIS 应为横向展开


In [ ]:


import numpy as np
import pandas as pd
import os
import rioxarray as rxr
from osgeo import gdal
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import PolynomialFeatures
from sklearn.metrics import accuracy_score
from sklearn.metrics import mean_absolute_error, root_mean_squared_error
from sklearn.preprocessing import PolynomialFeatures
from sklearn.linear_model import LinearRegression
from sklearn.pipeline import Pipeline, make_pipeline
import geopandas as gpd
import rasterio
from shapely.geometry import box
from shapely.geometry import Point
from rasterio.sample import sample_gen
import xgboost as xgb

from sklearn.svm import SVR
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.ensemble import RandomForestRegressor
from skopt import BayesSearchCV
from skopt.space import Integer, Real


In [75]:
def save_img(output_filename, reference_filename, array):
    # 打开参考数据
    reference = gdal.Open(reference_filename)
    
    # 获取地理变换参数和坐标系
    geo_transform = reference.GetGeoTransform()
    projection = reference.GetProjection()

    # 获取矩阵尺寸
    rows, cols = array.shape

    # 创建新的 TIFF 数据集
    driver = gdal.GetDriverByName('GTiff')
    out_data = driver.Create(output_filename, cols, rows, 1, gdal.GDT_Float32)

    # 设置地理变换和坐标系
    out_data.SetGeoTransform(geo_transform)
    out_data.SetProjection(projection)
    

    # 写入数据
    out_data.GetRasterBand(1).WriteArray(array)

    # 保存并关闭文件
    out_data.FlushCache()
    out_data = None

In [ ]:
path = 'G:\ICESat-2\FB_importance\wash_bay_2022_FB6_new5.tif'
output_path = 'G:\ICESat-2\FB_importance\wash_bay_2022_FB6_new5_masked.tif'
raster = rxr.open_rasterio(path)
mask = raster[-2]
raster[mask] = 0
raster_array = np.array(raster)
save_img(output_path, path, raster_array )

In [78]:
import rioxarray as rxr
import numpy as np
import rasterio

# 输入和输出路径
path = r'G:\ICESat-2\FB_importance\wash_bay_2022_FB6_new5.tif'
output_path = r'G:\ICESat-2\FB_importance\wash_bay_2022_FB6_new5_masked.tif'

# 读取raster
raster = rxr.open_rasterio(path, masked=True)  # 加 masked=True 会自动处理nodata
raster_array = raster.values.copy()  # 转为 numpy array，格式为 (bands, height, width)

# 取倒数第二个波段作为 mask
mask = raster_array[-2, :, :]  # shape 为 (H, W)

# 应用mask，将所有波段对应位置为0
raster_array[:, mask == 0] = 0  # 注意这里mask是布尔条件

# 保存图像
with rasterio.open(path) as src:
    profile = src.profile
    with rasterio.open(output_path, 'w', **profile) as dst:
        dst.write(raster_array)
